# 01 — Data Cleaning
**Dataset:** `apple_products_pricing_2020_2026.csv` (80,000 rows, 14 columns)
**Goal:** load, validate, document data quality issues, and export a cleaned file for the EDA/SQL stage.

See `data_dictionary.md` for the full column reference — this notebook re-derives and confirms those facts rather than assuming them.

In [20]:
import numpy as np
import pandas as pd

project_root = "/Users/vidhimishra/Desktop/apple-pricing-analysis"

## 1. Load and first look

In [21]:
df = pd.read_csv(project_root + '/data/apple_products_pricing_2020_2026.csv')

print(df.shape)
df.head()

(80000, 14)


,Date,Platform,Product_Category,Model_Name,Condition,Launch_Price_USD,Launch_Price_INR,Current_Price_USD,Current_Price_INR,Discount_Pct,Sale_Event,Stock_Status,Rating,Reviews_Count
0,2020-09-19,Flipkart,Watch,Apple Watch Series 6 (44mm),New,429,42042,435.81,43322.41,-1.6,NaN,In Stock,4.7,40
1,2020-09-20,Flipkart,Watch,Apple Watch Series 6 (44mm),New,429,42042,436.49,42320.43,-1.7,NaN,Out of Stock,4.6,84
2,2020-09-23,Amazon,Watch,Apple Watch Series 6 (44mm),New,429,42042,422.73,40879.36,1.5,NaN,In Stock,4.4,110
3,2020-09-23,Amazon,Watch,Apple Watch Series 6 (44mm),New,429,42042,425.00,42008.70,0.9,NaN,In Stock,4.8,111
4,2020-09-24,Amazon,Watch,Apple Watch Series 6 (44mm),New,429,42042,436.22,41984.28,-1.7,NaN,In Stock,4.7,35


In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80000 entries, 0 to 79999
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Date               80000 non-null  object 
 1   Platform           80000 non-null  object 
 2   Product_Category   80000 non-null  object 
 3   Model_Name         80000 non-null  object 
 4   Condition          80000 non-null  object 
 5   Launch_Price_USD   80000 non-null  int64  
 6   Launch_Price_INR   80000 non-null  int64  
 7   Current_Price_USD  80000 non-null  float64
 8   Current_Price_INR  80000 non-null  float64
 9   Discount_Pct       80000 non-null  float64
 10  Sale_Event         6649 non-null   object 
 11  Stock_Status       80000 non-null  object 
 12  Rating             80000 non-null  float64
 13  Reviews_Count      80000 non-null  int64  
dtypes: float64(4), int64(3), object(7)
memory usage: 8.5+ MB


In [23]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Date,80000,2130,2025-06-11,95,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Platform,80000,2,Flipkart,40043,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Product_Category,80000,4,iPhone,28589,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Model_Name,80000,31,iPhone 14 Pro 128GB,2734,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Condition,80000,2,New,59985,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Launch_Price_USD,80000.0,NaN,NaN,NaN,963.959125,470.086413,329.0,599.0,799.0,1199.0,1999.0
Launch_Price_INR,80000.0,NaN,NaN,NaN,94467.99425,46068.468464,32242.0,58702.0,78302.0,117502.0,195902.0
Current_Price_USD,80000.0,NaN,NaN,NaN,782.769855,461.67397,109.93,432.93,699.74,989.1125,2038.97
Current_Price_INR,80000.0,NaN,NaN,NaN,74628.342439,45117.872172,9157.68,41686.835,67324.02,96568.1175,203668.71
Discount_Pct,80000.0,NaN,NaN,NaN,21.418826,16.69731,-2.0,6.7,21.3,36.8,73.1


## 2. Missing values

Only `Sale_Event` has nulls, and that's expected — most days aren't sale days. We confirm the null rate matches
what we found during profiling (~91.7%) and then fill blanks with an explicit `'No Event'` label rather than
leaving them as NaN, so groupby/aggregation later doesn't silently drop them.

In [24]:
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})

,missing_count,missing_pct
Date,0,0.00
Platform,0,0.00
Product_Category,0,0.00
Model_Name,0,0.00
Condition,0,0.00
Launch_Price_USD,0,0.00
Launch_Price_INR,0,0.00
Current_Price_USD,0,0.00
Current_Price_INR,0,0.00
Discount_Pct,0,0.00


In [25]:
assert df.drop(columns=['Sale_Event']).isna().sum().sum() == 0, "Unexpected nulls outside Sale_Event"

df['Sale_Event'] = df['Sale_Event'].fillna('No Event')
df['Sale_Event'].value_counts()

Sale_Event
No Event                 73351
Black Friday              2497
Big Billion Days          1579
Great Indian Festival     1504
Prime Day                 1069
Name: count, dtype: int64

## 3. Duplicate check

Two checks: exact full-row duplicates, and duplicates on what looked like the natural grain
(`Date`, `Platform`, `Model_Name`, `Condition`).

In [26]:
print('Exact duplicate rows:', df.duplicated().sum())

grain_cols = ['Date', 'Platform', 'Model_Name', 'Condition']
grain_dupes = df.duplicated(subset=grain_cols, keep=False).sum()
print(f'Rows sharing the same {grain_cols} combination:', grain_dupes)

Exact duplicate rows: 0
Rows sharing the same ['Date', 'Platform', 'Model_Name', 'Condition'] combination: 37274


**Finding:** there are zero exact duplicate rows, but tens of thousands of rows share the same
date/platform/model/condition combination with *different* prices. That means the dataset is **not**
one row per listing per day — there are multiple price snapshots recorded for the same combination on
the same date (e.g. intraday price changes, or an unlabeled seller/listing dimension).

This matters for anything downstream that assumes "one price per model per day": if you build a daily
time series, decide explicitly whether to take the mean, min, max, or last observation per group —
don't just plot the raw rows as if they were unique.

In [27]:
example_key = (df[df.duplicated(subset=grain_cols, keep=False)]
               .groupby(grain_cols).size()
               .sort_values(ascending=False)
               .head(3))
example_key

Date        Platform  Model_Name       Condition
2026-01-30  Flipkart  iPhone 17 128GB  New          12
2026-07-13  Flipkart  iPhone 17 128GB  New          11
2026-05-31  Flipkart  iPhone 17 128GB  New          10
dtype: int64

## 4. Sanity check: is `Launch_Price` really fixed per model?

The data dictionary assumes `Launch_Price_USD`/`Launch_Price_INR` are constant reference values per model.
Confirm that before treating them as a fixed baseline anywhere else.

In [28]:
launch_price_variation = df.groupby('Model_Name')['Launch_Price_USD'].nunique()
inconsistent_models = launch_price_variation[launch_price_variation > 1]

print('Models with more than one Launch_Price_USD value:', len(inconsistent_models))
inconsistent_models

Models with more than one Launch_Price_USD value: 0


Series([], Name: Launch_Price_USD, dtype: int64)

## 5. Sanity check: does `Discount_Pct` actually match `Launch_Price` vs `Current_Price`?

Recompute the discount from the two price columns and compare to the stored value, allowing for
rounding (the stored column looks rounded to 1 decimal place).

In [29]:
recomputed_discount = (df['Launch_Price_USD'] - df['Current_Price_USD']) / df['Launch_Price_USD'] * 100
discount_diff = (recomputed_discount - df['Discount_Pct']).abs()

print('Max difference (recomputed vs stored):', round(discount_diff.max(), 4))
print('Rows differing by more than rounding tolerance (0.15):', (discount_diff > 0.15).sum())

Max difference (recomputed vs stored): 0.0514
Rows differing by more than rounding tolerance (0.15): 0


**Finding:** the recomputed discount matches the stored `Discount_Pct` within rounding tolerance for every row —
the column is reliable and reproducible, not an unexplained black box. Good to know before quoting it in analysis.

Also worth remembering: **negative `Discount_Pct` means the current price is *above* launch price** (a markup),
not a discount. About 10.8% of rows fall into this category — that's an anomaly to investigate in `02_eda.ipynb`,
not something to clean away here.

## 6. Parse dates and add derived time fields

In [30]:
df['Date'] = pd.to_datetime(df['Date'])
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Year_Month'] = df['Date'].dt.to_period('M').astype(str)
df["Quarter"] = df["Date"].dt.quarter
df["Weekday"] = df["Date"].dt.day_name()

df[['Date', 'Year', 'Month', 'Year_Month', 'Quarter', 'Weekday']].head()

,Date,Year,Month,Year_Month,Quarter,Weekday
0,2020-09-19,2020,9,2020-09,3,Saturday
1,2020-09-20,2020,9,2020-09,3,Sunday
2,2020-09-23,2020,9,2020-09,3,Wednesday
3,2020-09-23,2020,9,2020-09,3,Wednesday
4,2020-09-24,2020,9,2020-09,3,Thursday


## 7. Set categorical dtypes

Reduces memory and makes groupby operations faster/cleaner for the low-cardinality text columns.

In [31]:
categorical_cols = ['Platform', 'Product_Category', 'Model_Name', 'Condition', 'Sale_Event', 'Stock_Status']
for c in categorical_cols:
    df[c] = df[c].astype('category')

df.dtypes

Date                 datetime64[ns]
Platform                   category
Product_Category           category
Model_Name                 category
Condition                  category
Launch_Price_USD              int64
Launch_Price_INR              int64
Current_Price_USD           float64
Current_Price_INR           float64
Discount_Pct                float64
Sale_Event                 category
Stock_Status               category
Rating                      float64
Reviews_Count                 int64
Year                          int32
Month                         int32
Year_Month                   object
Quarter                       int32
Weekday                      object
dtype: object

## 8. Price drop in absolute USD terms (Launch vs Current)

In [32]:
# Price drop in absolute USD terms (Launch vs Current)
df["Price_Drop_USD"] = df["Launch_Price_USD"] - df["Current_Price_USD"]
df["Price_Drop_Pct"] = ((df["Launch_Price_USD"] - df["Current_Price_USD"]) / df["Launch_Price_USD"] * 100).round(2)

## 8. Flag markup rows

Create an explicit boolean flag for the "price above launch" anomaly found in Section 5, so it's easy
to filter/group on in the next notebook without re-deriving it.

In [33]:
df['Is_Markup'] = df['Discount_Pct'] < 0

df['Is_Markup'].value_counts(normalize=True).round(4)

Is_Markup
False    0.8916
True     0.1084
Name: proportion, dtype: float64

## 9. Final checks before export

In [34]:
print('Shape:', df.shape)
print()
print('Remaining nulls:')
print(df.isna().sum()[df.isna().sum() > 0])
print()
df.sample(5, random_state=42)

Shape: (80000, 22)

Remaining nulls:
Series([], dtype: int64)



,Date,Platform,Product_Category,Model_Name,Condition,Launch_Price_USD,Launch_Price_INR,Current_Price_USD,Current_Price_INR,Discount_Pct,...,Rating,Reviews_Count,Year,Month,Year_Month,Quarter,Weekday,Price_Drop_USD,Price_Drop_Pct,Is_Markup
47044,2025-04-11,Amazon,iPhone,iPhone 16 Pro 256GB,New,1099,107702,1097.47,107480.49,0.1,...,4.3,1050,2025,4,2025-04,2,Friday,1.53,0.14,False
44295,2025-02-27,Amazon,iPad,iPad Pro 12.9-inch (M2) 256GB,New,1199,117502,1001.40,96246.04,16.5,...,4.4,3983,2025,2,2025-02,1,Thursday,197.60,16.48,False
74783,2026-05-21,Flipkart,iPhone,iPhone 15 128GB,Renewed/Refurbished,799,78302,432.79,36865.41,45.8,...,4.4,1744,2026,5,2026-05,2,Thursday,366.21,45.83,False
70975,2026-03-29,Amazon,iPhone,iPhone 16 Pro 256GB,New,1099,107702,999.85,97344.40,9.0,...,4.8,1570,2026,3,2026-03,1,Sunday,99.15,9.02,False
46645,2025-04-05,Flipkart,Watch,Apple Watch Series 9 (45mm),New,429,42042,329.26,32091.76,23.2,...,4.5,2401,2025,4,2025-04,2,Saturday,99.74,23.25,False


## 10. Export cleaned dataset

Saved as a separate file — the original raw CSV is never overwritten, so the cleaning steps stay reproducible
and auditable.

In [35]:
df.to_csv(project_root + '/data/apple_products_pricing_cleaned.csv', index=False)
print('Saved: ' + project_root + '/data/apple_products_pricing_cleaned.csv')
print('Final shape:', df.shape)

Saved: /Users/vidhimishra/Desktop/apple-pricing-analysis/data/apple_products_pricing_cleaned.csv
Final shape: (80000, 22)


## Summary of cleaning decisions

- `Sale_Event` nulls (91.7% of rows) filled with `'No Event'` — this was an expected non-event day, not missing data.
- No exact duplicate rows. However, `Date` + `Platform` + `Model_Name` + `Condition` is **not** a unique key — decide
  on an aggregation method (mean/last/etc.) before building any daily time series.
- `Launch_Price_USD` confirmed constant per model — safe to use as a fixed reference price.
- `Discount_Pct` confirmed to be a faithful (rounded) recomputation of `Launch_Price` vs `Current_Price` — trustworthy as-is.
- Added `Year`, `Month`, `Year_Month`, and `Is_Markup` as derived columns for the EDA stage.
- No rows were dropped in this notebook — the "anomaly" (markup rows) is a finding to analyze in `02_eda.ipynb`, not a defect to remove.